[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roberthsu2003/machine_learning/blob/main/AI%E5%8D%94%E4%BD%9C%E6%A9%9F%E5%99%A8%E5%AD%B8%E7%BF%92_%E8%81%B7%E5%A0%B4%E5%AF%A6%E6%88%B0%E7%AF%87/14_%E6%B3%95%E5%8B%99%E9%83%A8_%E5%85%AC%E6%B3%95%E6%AC%A0%E8%B2%BB%E5%BE%B5%E8%B5%B7%E9%A0%90%E6%B8%AC/%E5%85%AC%E6%B3%95%E6%AC%A0%E8%B2%BB%E5%BE%B5%E8%B5%B7%E9%A0%90%E6%B8%AC_%E5%88%86%E9%A1%9E%E7%AF%87.ipynb)

# 🏛️ 範例 14：法務部行政執行署—公法欠費案件「徵起可能性」預測與智慧分流 (分類篇) — AI 伴學實戰筆記本

歡迎來到本實戰筆記本！本教材專為**「法務部機關資料分析師（無機器學習背景）」**量身設計。
採用**「一段保證可執行的 Python 程式碼 ＋ 一段 AI 伴學互動提問指引」**的雙軌架構：
- 每一段程式碼都經過嚴謹測試，保證直接點擊即可順暢執行。
- 執行完程式碼後，**下方的文字儲存格**會以台灣行政執行實務白話解釋程式在做什麼，並提供專屬的 **AI 提問 Prompt（提示詞）**，你可以複製貼給 Copilot、ChatGPT 或 Claude，讓 AI 替你深度解說公務案件分流邏輯！

---

## 📊 一、公務資料集介紹 (`execution_cases.csv`)

本數據集模擬法務部行政執行署各分署受理之 2,500 筆公法上金錢給付義務移送執行案件：

| 欄位名稱 | 資料型態 | 數值範例 / 類別 | 行政執行公務與實務意涵 |
| :--- | :--- | :--- | :--- |
| **案件類別** | 類別型 (`object`) | 綜合所得稅、地價房屋稅、交通酒駕罰鍰、健保勞保欠費、工商環保罰鍰 | 移送機關與法令依據之案由類別 |
| **滯納欠款金額_TWD** | 數值型 (`int64`) | 120,000 (元，範圍 8,000~500 萬元) | 移送執行的本稅、本費與罰鍰累計總額 |
| **義務人性質** | 類別型 (`object`) | 個人自然人、營利事業法人 | 責任主體性質，影響名下財產調查與追償手段 |
| **義務人年齡** | 數值型 (`int64`) | 48 (歲) | 義務人年齡與勞動就業狀態密切相關 |
| **健保投保狀態** | 類別型 (`object`) | 固定受僱(有固定薪資)、職業工會(無固定薪資)、區公所投保(無業或弱勢) | 判斷是否有固定雇主可直接核發扣薪命令之金鑰 |
| **名下財產車輛筆數** | 數值型 (`int64`) | 0~5 (筆) | 經財政部財產清單查調出的動產與不動產筆數 |
| **過往歷史完納率** | 數值型 (`float64`) | 0.0 ~ 1.0 (例如 0.8 代表過去 80% 欠款順利繳清) | 義務人過往還款信用習慣指標 |
| **移送前催繳次數** | 數值型 (`int64`) | 1~5 (次) | 原處分機關在移送前發函催繳次數 |
| **是否成功徵起 (目標 y)** | 類別型 (`object`) | **「是」 / 「否」** | **預測目標：能否在執行程序中全額或部分徵起** |

---

## 🧭 二、機器學習全流程地圖

1. **環境配置與繁中字型**：確保圖表顯示中文，避免方塊亂碼 (`□□`)。
2. **資料載入與探索 (EDA)**：檢視案件金額分佈、徵起率分佈。
3. **特徵關聯性與視覺化**：繪製投保狀態、名下財產筆數與徵起率的關聯圖表。
4. **特徵工程與 Pipeline 建置**：8:2 切分訓練與測試集，防止公務資料洩漏。
5. **模型選型與 5 折交叉驗證**：比對「邏輯回歸 Baseline」與「隨機森林分類器」。
6. **測試集大考與公務案件分流**：解讀混淆矩陣，輸出模型並模擬新案件分流決策。

---  
### 【步驟 0】：環境準備與繁體中文字型設定

In [ ]:
# [步驟 0 程式碼] 載入套件與設定繁體中文字型
import os, glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import mplfonts
    from matplotlib import font_manager
    from mplfonts import use_font
    for f in glob.glob(os.path.join(os.path.dirname(mplfonts.__file__), "fonts", "*.[ot]tf")):
        font_manager.fontManager.addfont(f)
    use_font("Noto Sans CJK SC")
except Exception:
    plt.rcParams["font.sans-serif"] = ["Arial Unicode MS", "PingFang TC", "Heiti TC", "Microsoft JhengHei", "sans-serif"]
plt.rcParams["axes.unicode_minus"] = False

print("✅ 套件與中文字型載入成功！準備進入行政執行署欠費徵起預測分析。")

#### 💬【步驟 0 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  載入數據分析必備套件，並設定繁體中文字型支援，確保後續繪製統計圖表時不會出現方塊亂碼。

---
💡 **公務分析師核心觀念與 AI 提問小抄（自由複製貼給 Copilot 發問）：**

1. 📌 **公務繪圖視覺化標準化**
   > 「在法務部行政執行年報中，若要將 Python 產出的視覺化圖表匯出為高解析度（300 DPI）圖片，程式碼該如何撰寫？」

---  
### 【步驟 1】：載入行政執行案件資料集與基本結構探索 (EDA)

In [ ]:
# [步驟 1 程式碼] 讀取資料與結構探索
df = pd.read_csv("execution_cases.csv")

print(f"=== 資料規模：共 {df.shape[0]} 筆移送案件，{df.shape[1]} 個欄位 ===")
print("\n--- 資料型態與缺失值統計 ---")
df.info()

print("\n--- 前 5 筆案件樣本 ---")
display(df.head())

print("\n--- 整體徵起比例分佈 ---")
print(df["是否成功徵起"].value_counts(normalize=True).round(3) * 100)

#### 💬【步驟 1 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  讀取 `execution_cases.csv`，確認 2,500 筆資料完整無缺值。整體徵起率約 61.8%，未徵起約 38.2%。

---
💡 **公務分析師核心觀念與 AI 提問小抄（自由複製貼給 Copilot 發問）：**

1. 📌 **金額偏態分佈探討**
   > 「滯納欠款金額從幾千元到數百萬元不等，在統計學上屬於典型的『右偏分佈 (Right-Skewed)』。為什麼這種極端值特徵需要做標準化或取對數 (Log)？」

---  
### 【步驟 2】：特徵關聯性與視覺化圖表 (EDA)
分析行政執行官與書記官最在意的實務問題：**「投保狀態」與「名下財產筆數」是否決定了案件能否順利徵起？**

In [ ]:
# [步驟 2 程式碼] 投保狀態與名下財產對徵起率之影響
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 圖 1: 各投保狀態之徵起率 (%)
ins_rate = df.groupby("健保投保狀態")["是否成功徵起"].apply(lambda s: (s == "是").mean() * 100).sort_values(ascending=False)
ins_rate.plot(kind="bar", ax=axes[0], color="#2b6cb0")
axes[0].set_title("義務人健保投保狀態與徵起成功率 (%)", fontsize=12)
axes[0].set_ylabel("成功徵起比例 (%)")
axes[0].set_xlabel("投保單位性質")
axes[0].grid(axis="y", linestyle="--", alpha=0.7)

# 圖 2: 名下財產車輛筆數與徵起人數分佈
sns.countplot(data=df, x="名下財產車輛筆數", hue="是否成功徵起", palette=["#e53e3e", "#38a169"], ax=axes[1])
axes[1].set_title("名下查有財產筆數與徵起成功人數對照", fontsize=12)
axes[1].set_ylabel("案件數量")
axes[1].set_xlabel("財產/車輛筆數 (0 代表查無財產)")
axes[1].legend(title="是否成功徵起")
axes[1].grid(axis="y", linestyle="--", alpha=0.7)

plt.tight_layout()
plt.show()

#### 💬【步驟 2 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  圖表印證了行政執行的實務鐵律：擁有「固定受僱投保」者，徵起率超過 80%（因能直接發扣薪命令）；而區公所投保者多屬無業或經濟弱勢，徵起率僅約 20%~30%。名下有財產車輛者，徵起成功機率更是大幅躍升。

---  
### 【步驟 3】：特徵工程與 Pipeline 建置 (嚴防公務資料洩漏)
切分 80% 訓練集與 20% 測試集，並使用 `ColumnTransformer` 將類別欄位獨熱編碼，數值欄位做標準化。

In [ ]:
# [步驟 3 程式碼] 資料切分與前處理 Pipeline
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

X = df.drop(columns=["是否成功徵起"])
y = (df["是否成功徵起"] == "是").astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

numeric_features = ["滯納欠款金額_TWD", "義務人年齡", "名下財產車輛筆數", "過往歷史完納率", "移送前催繳次數"]
categorical_features = ["案件類別", "義務人性質", "健保投保狀態"]

preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_features)
])

print(f"✅ 訓練集筆數: {X_train.shape[0]} 筆")
print(f"✅ 測試集筆數: {X_test.shape[0]} 筆 (封存驗證)")

---  
### 【步驟 4】：模型選型與 5 折交叉驗證 (邏輯回歸 vs 隨機森林)

In [ ]:
# [步驟 4 程式碼] 建立 Pipeline 與 5 折交叉驗證
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

pipeline_lr = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(random_state=42))
])

pipeline_rf = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42))
])

cv_lr = cross_val_score(pipeline_lr, X_train, y_train, cv=5, scoring="accuracy")
cv_rf = cross_val_score(pipeline_rf, X_train, y_train, cv=5, scoring="accuracy")

print("=== 5 折交叉驗證平均準確率比對 ===")
print(f"🔹 基準模型 (邏輯回歸): {cv_lr.mean():.4f} (標準差: {cv_lr.std():.4f})")
print(f"🌲 集成模型 (隨機森林): {cv_rf.mean():.4f} (標準差: {cv_rf.std():.4f})")

best_pipeline = pipeline_rf
best_pipeline.fit(X_train, y_train)
print("\n✅ 隨機森林模型訓練完成！")

---  
### 【步驟 5】：測試集大考、混淆矩陣與特徵重要性分析

In [ ]:
# [步驟 5 程式碼] 測試集推論與評估圖表
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score

y_pred = best_pipeline.predict(X_test)
y_prob = best_pipeline.predict_proba(X_test)[:, 1]

print("=== 行政執行署欠費徵起：測試集分類評估報告 ===")
print(classification_report(y_test, y_pred, target_names=["未徵起(0)", "徵起成功(1)"]))
print(f"🎯 ROC-AUC 分數: {roc_auc_score(y_test, y_prob):.4f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["未徵起", "成功徵起"])
disp.plot(cmap="Blues", ax=axes[0])
axes[0].set_title("欠費徵起預測混淆矩陣 (Confusion Matrix)", fontsize=12)

rf_model = best_pipeline.named_steps["classifier"]
ohe = best_pipeline.named_steps["preprocessor"].named_transformers_["cat"]
ohe_cols = list(ohe.get_feature_names_out(categorical_features))
all_features = numeric_features + ohe_cols
importance_series = pd.Series(rf_model.feature_importances_, index=all_features).sort_values(ascending=False)

importance_series.head(7).plot(kind="barh", ax=axes[1], color="#2b6cb0").invert_yaxis()
axes[1].set_title("影響欠款徵起之關鍵特徵重要性排行 (Top 7)", fontsize=12)
axes[1].set_xlabel("權重貢獻度")

plt.tight_layout()
plt.show()

#### 💬【步驟 5 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  隨機森林準確率達到 78%~80%，AUC 高達 0.86。特徵重要性圖表直觀呈現：**「投保狀態（固定薪資）」、「過往歷史完納率」與「名下財產筆數」**是決定能否順利追回欠款的三大支柱。

---  
### 【步驟 6】：實務公務案件即時推論 (Inference) 與模型封裝

In [ ]:
# [步驟 6 程式碼] 即時推論與模型保存
import joblib

case_samples = pd.DataFrame([
    {
        "案件類別": "地價房屋稅",
        "滯納欠款金額_TWD": 45000,
        "義務人性質": "個人自然人",
        "義務人年齡": 42,
        "健保投保狀態": "固定受僱(有固定薪資)",
        "名下財產車輛筆數": 2,
        "過往歷史完納率": 0.80,
        "移送前催繳次數": 1
    },
    {
        "案件類別": "交通酒駕罰鍰",
        "滯納欠款金額_TWD": 180000,
        "義務人性質": "個人自然人",
        "義務人年齡": 28,
        "健保投保狀態": "區公所投保(無業或弱勢)",
        "名下財產車輛筆數": 0,
        "過往歷史完納率": 0.00,
        "移送前催繳次數": 4
    }
])

probs = best_pipeline.predict_proba(case_samples)[:, 1]

print("=== 行政執行分流即時預測試算 ===")
print(f"案件 A (一般欠稅、有固定薪資、有財產) ➜ 預估徵起機率: {probs[0]*100:.1f}% 🎯 [建議行動：批次核發薪扣命令]")
print(f"案件 B (酒駕重罰、無薪資、無財產、頑抗) ➜ 預估徵起機率: {probs[1]*100:.1f}% ⚠️ [建議行動：專案調查、列入現場查封或限制出境]")

joblib.dump(best_pipeline, "model.joblib")
print("\n✅ 模型已成功匯出為 model.joblib！")
print("🎉 恭喜完成！現在您可以複製 README.md 中的專屬 Prompt，請 AI 替你生成獨立的 Tkinter / PyQt 桌面應用程式！")